In [1]:
# Instala todas as dependencias
# O ponto de exclamacao (!) roda comandos de terminal dentro do Jupyter
# chamadas HTTP (APIs REST)
!pip install requests~=2.31
# retry com backoff exponencial
!pip install tenacity~=8.2 
# carrega variáveis de ambiente do arquivo .env
!pip install python-dotenv~=1.0 
# framework de criação de APIs
!pip install fastapi~=0.141 
# O uvicorn é o servidor ASGI que efetivamente executa sua API FastAPI.
!pip install uvicorn[standard]~=0.35 


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



   ---------------------------------------- 0/5 [websockets]
   ---------------------------------------- 0/5 [websockets]
   ---------------------------------------- 0/5 [websockets]
   ---------------------------------------- 0/5 [websockets]
   ---------------------------------------- 0/5 [websockets]
   ---------------------------------------- 0/5 [websockets]
   ---------------------------------------- 0/5 [websockets]
   ---------------------------------------- 0/5 [websockets]
   ---------------------------------------- 0/5 [websockets]
   ---------------------------------------- 0/5 [websockets]
   ---------------------------------------- 0/5 [websockets]
   ---------------------------------------- 0/5 [websockets]
   ---------------------------------------- 0/5 [websockets]
   ---------------------------------------- 0/5 [websockets]
   ---------------------------------------- 0/5 [websockets]
   ---------------------------------------- 0/5 [websockets]
   --------------------


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


---
## Estrutura de pastas do projeto

Vamos criar a estrutura de diretorios do projeto agora.

```
mlops-pipeline/
    src/data/       <- codigo de coleta (este episodio)
    tests/          <- testes automatizados (este episodio)
    scripts/        <- utilitarios (servidor fake, geracao de dados)
    data/raw/       <- dados brutos (NUNCA editados a mao)
    data/mock/      <- dados sinteticos para testes
```

> **Regra de ouro:** a pasta `data/raw/` e sagrada.
> Tudo que chegar aqui fica intocado. Se precisarmos processar, criamos uma nova pasta.
> O dado bruto e nosso seguro, se tudo der errado, voltamos a ele.

In [2]:
from pathlib import Path

# Cria a estrutura completa de pastas do projeto
# parents=True: cria pastas intermediarias se nao existirem
# exist_ok=True: nao da erro se a pasta ja existir
pastas = [
    './src/data',
    './tests',
    './scripts',
    './data/mock',
]

for pasta in pastas:
    Path(pasta).mkdir(parents=True, exist_ok=True)
    print(f'Criada: {pasta}/')

# Cria os arquivos __init__.py necessarios para o Python reconhecer os modulos
Path('./src/__init__.py').touch()
Path('./src/data/__init__.py').touch()
Path('./tests/__init__.py').touch()

print('\nEstrutura criada com sucesso!')

Criada: ./src/data/
Criada: ./tests/
Criada: ./scripts/
Criada: ./data/mock/

Estrutura criada com sucesso!


---
## Parte 1 — Coleta via API REST
**Arquivo:** `src/data/collect_api.py`

Vamos construir um coletor de API com tres caracteristicas essenciais:
1. **Autenticacao** — token no header
2. **Paginacao** — busca pagina a pagina ate acabar
3. **Retry com backoff** — tenta de novo se a API falhar temporariamente

### Conceitos que vamos usar

- **Retry:** tentar de novo automaticamente apos uma falha
- **Backoff exponencial:** esperar cada vez mais entre tentativas (1s, 2s, 4s...)
- **Jitter:** adicionar aleatoriedade ao backoff para evitar thundering herd
- **Thundering herd:** quando varios clientes tentam reconectar ao mesmo tempo e derrubam o servidor

In [3]:
# === PASSO 1: Imports e constantes ===
# Cada import tem uma funcao especifica — nunca importe o que nao vai usar

import logging        # sistema de log — melhor que print() em producao
import os             # acesso a variaveis de ambiente
import requests       # chamadas HTTP para APIs REST
import pandas as pd   # manipulacao de dados tabulares
from pathlib import Path  # manipulacao de caminhos de arquivo (cross-platform)
from tenacity import (    # retry inteligente com backoff
    retry,
    stop_after_attempt,
    wait_random_exponential,
    retry_if_exception,
    before_sleep_log,
)

# Configura o logger para este modulo
# __name__ = 'src.data.collect_api' — identifica de onde vem o log
logger = logging.getLogger(__name__)

# Codigos HTTP que justificam uma nova tentativa (erros temporarios)
# 429 = Too Many Requests | 500/502/503/504 = erros do servidor
# 401 (Unauthorized) e 404 (Not Found) NAO estao aqui — sao erros permanentes
RETRYABLE_CODES = (429, 500, 502, 503, 504)

print('Imports carregados com sucesso!')
print(f'Codigos com retry: {RETRYABLE_CODES}')

Imports carregados com sucesso!
Codigos com retry: (429, 500, 502, 503, 504)


In [4]:
"""
collect_api.py — Coleta de dados via API REST
==============================================
Responsável por:
- Fazer requisições HTTP com autenticação Bearer token
- Navegar automaticamente por todas as páginas de resultados (paginação)
- Fazer retry automático com backoff exponencial em caso de falha temporária
- Salvar o resultado em formato Parquet (eficiente e tipado)

Conceitos ensinados neste módulo (Ep 01, Parte 1):
- O que é uma API REST e como autenticar com Bearer token
- O que é paginação e por que APIs a usam
- O que é retry, backoff exponencial e jitter
- O que é um worker e o problema do thundering herd
- Por que usar Session em vez de requests direto
- Por que salvar em Parquet em vez de CSV
"""

import logging
import os

import pandas as pd
import requests
from pathlib import Path
from tenacity import (
    before_sleep_log,
    retry,
    retry_if_exception,
    stop_after_attempt,
    wait_random_exponential,
)

# Logger específico deste módulo.
# __name__ resolve para "src.data.collect_api" — facilita filtrar logs por módulo.
logger = logging.getLogger(__name__)

# Códigos HTTP que indicam falha TEMPORÁRIA no servidor (vale tentar de novo):
#   429 = Too Many Requests  → você enviou requisições rápido demais (rate limit)
#   500 = Internal Server Error → bug pontual no servidor
#   502 = Bad Gateway           → servidor intermediário (proxy/load balancer) com problema
#   503 = Service Unavailable   → servidor sobrecarregado ou em manutenção
#   504 = Gateway Timeout       → servidor não respondeu no tempo limite
#
# Erros que NÃO estão aqui (ex: 401 Unauthorized, 404 Not Found) são permanentes:
# tentar de novo não vai resolver — são erros de configuração.
RETRYABLE_CODES = (429, 500, 502, 503, 504)

class APICollector:
    """Coletor robusto para APIs REST com paginação e retry automático.

    Uso básico:
        collector = APICollector(
            base_url="https://api.empresa.com/v1",
            token="meu_token",
        )
        df = collector.collect("customers", Path("data/raw/customers.parquet"))

    Parâmetros de retry:
        max_attempts:  número máximo de tentativas (padrão: 5)
        min_wait_secs: tempo mínimo de espera entre tentativas (padrão: 1s)
        max_wait_secs: tempo máximo de espera entre tentativas (padrão: 60s)
    """

    def __init__(
        self,
        base_url: str,
        token: str,
        max_attempts: int = 5,
        min_wait_secs: float = 1.0,
        max_wait_secs: float = 60.0,
    ):
        """Inicializa o coletor e configura a sessão HTTP.

        Args:
            base_url:      URL raiz da API. Ex: "https://api.empresa.com/v1"
                           A barra final é removida automaticamente se existir.
            token:         Bearer token de autenticação.
                           Gerado pelo time de backend ou painel da API.
            max_attempts:  Quantas vezes tentar uma requisição que falhou.
                           Inclui a tentativa original (não apenas os retries).
            min_wait_secs: Tempo mínimo de espera antes de tentar novamente.
            max_wait_secs: Teto de espera — nunca espera mais que isso.
        """
        self.base_url = base_url.rstrip("/")
        self.max_attempts = max_attempts
        self.min_wait_secs = min_wait_secs
        self.max_wait_secs = max_wait_secs

        # requests.Session() mantém uma conexão TCP aberta e a reutiliza.
        # Sem Session: cada .get() abre nova conexão → lento (handshake TCP + TLS).
        # Com Session:  a conexão é reaproveitada entre requisições → 3-5x mais rápido.
        self.session = requests.Session()

        # Headers padrão enviados em TODAS as requisições desta sessão.
        # Authorization: o servidor usa para verificar sua identidade.
        # Accept: diz ao servidor o formato que esperamos na resposta.
        self.session.headers.update(
            {
                "Authorization": f"Bearer {token}",
                "Accept": "application/json",
            }
        )

    def _is_retryable_exception(self, exc: BaseException) -> bool:
        """Determina se uma exceção justifica fazer retry.
    
        Regra:
        - ConnectionError e Timeout → sempre retryable (falha de rede/infra)
        - HTTPError com código 429 ou 5xx → retryable (falha temporária do servidor)
        - HTTPError com código 4xx (exceto 429) → NÃO retryable (erro permanente)
        - Qualquer outra exceção → NÃO retryable (bug no código)
    
        Args:
            exc: A exceção que foi levantada.
    
        Returns:
            True se deve fazer retry, False se deve propagar imediatamente.
        """
        if isinstance(exc, (requests.ConnectionError, requests.Timeout)):
            return True
        if isinstance(exc, requests.HTTPError):
            # exc.response pode ser None em alguns casos de erro de rede
            if exc.response is not None:
                return exc.response.status_code in RETRYABLE_CODES
        return False

    def _make_retry_decorator(self):
        """Cria o decorator de retry com a configuração desta instância.

        Separado em método para permitir criar instâncias com políticas diferentes
        em tempo de execução (ex: ambiente de testes com 0s de espera).

        Returns:
            Decorator da tenacity configurado com os parâmetros desta instância.
        """
        return retry(
            reraise=True,
            # reraise=True: ao esgotar as tentativas, re-lança a exceção original.
            # Sem isso: levantaria uma RetryError genérica, difícil de depurar.
            # Com isso:  você vê exatamente qual HTTPError / ConnectionError ocorreu.

            stop=stop_after_attempt(self.max_attempts),
            # Para após max_attempts tentativas no total.
            # Ex: max_attempts=5 → 1 original + 4 retries.

            wait=wait_random_exponential(
                min=self.min_wait_secs,
                max=self.max_wait_secs,
            ),
            # wait_random_exponential = backoff exponencial + jitter embutido.
            #
            # Backoff exponencial: o tempo de espera dobra a cada falha.
            # Sequência base (min=1, max=60): ~1s, ~2s, ~4s, ~8s, ~16s
            #
            # Jitter: variação aleatória adicionada a cada valor.
            # Por que jitter? Imagine 50 workers que todos falham ao mesmo tempo
            # (ex: o servidor reiniciou). Sem jitter, todos esperam EXATAMENTE
            # 4 segundos e disparam 50 requisições simultâneas novamente,
            # sobrecarregando o servidor de novo — isso é o "thundering herd".
            # Com jitter, cada worker espera um tempo ligeiramente diferente,
            # espalhando as tentativas no tempo e protegendo o servidor.

            retry=retry_if_exception(self._is_retryable_exception),
            # Define QUAIS exceções acionam o retry.
            #
            # retry_if_exception(fn): faz retry se fn(exceção) retornar True.
            # Usamos uma função customizada para distinguir erros temporários
            # de erros permanentes dentro da mesma classe HTTPError:
            #
            #   RETRYABLE (vale tentar de novo):
            #     ConnectionError — falha de rede (sem internet, DNS falhou)
            #     Timeout         — servidor não respondeu no tempo limite
            #     HTTPError 429   — rate limit (você enviou requisições demais)
            #     HTTPError 5xx   — erros de servidor (500, 502, 503, 504)
            #
            #   NÃO RETRYABLE (erro permanente, tentar de novo não resolve):
            #     HTTPError 4xx   — 401 Unauthorized, 403 Forbidden, 404 Not Found
            #                       Esses erros indicam problema de configuração:
            #                       token errado, endpoint errado, recurso inexistente.
            #                       Tentar de novo apenas desperdiça tempo.
            #
            # Esta distinção é CRÍTICA: sem ela, um 404 causaria 5 tentativas
            # inúteis antes de finalmente falhar — e você perderia 1+ minuto.

            before_sleep=before_sleep_log(logger, logging.WARNING),
            # Loga automaticamente antes de cada espera, ex:
            # "Retrying _get_page in 4.3s as it raised HTTPError: 503 Server Error"
            # Sem isso, o retry seria silencioso — difícil saber o que está acontecendo.
        )

    def _get_page(self, url: str, params: dict) -> dict:
        """Faz uma única requisição GET sem lógica de retry.

        Por que separar a requisição do retry?
        1. Testabilidade: podemos mockar _get_page nos testes sem precisar
           lidar com a lógica de retry.
        2. Responsabilidade única: _get_page só sabe fazer a requisição;
           quem decide se vai tentar de novo é o decorator em collect().
        3. Clareza: ao ler o código, fica óbvio onde a requisição acontece.

        Args:
            url:    URL completa do endpoint.
            params: Parâmetros de query string (paginação, filtros, etc.).

        Returns:
            Dicionário Python com o corpo da resposta JSON.

        Raises:
            requests.HTTPError:       se o servidor retornar código de erro.
            requests.ConnectionError: se não for possível conectar.
            requests.Timeout:         se o servidor não responder em 20 segundos.
        """
        response = self.session.get(url, params=params, timeout=20)
        # timeout=20: se o servidor não enviar nenhum byte em 20 segundos, lança Timeout.
        # Sem timeout, seu programa pode ficar travado indefinidamente aguardando.

        # Para qualquer código de erro (401, 403, 404...):
        response.raise_for_status()

        # .json() deserializa a string JSON da resposta para dict/list Python.
        return response.json()

    def collect(
        self,
        endpoint: str,
        output_path: Path,
        page_size: int = 500,
    ) -> pd.DataFrame:
        """Coleta todas as páginas de um endpoint e salva em Parquet.

        Fluxo:
            1. Faz GET na página 1
            2. Adiciona os registros à lista acumuladora
            3. Verifica se há próxima página (campo has_next na resposta)
            4. Se sim, incrementa page e repete
            5. Ao terminar, converte para DataFrame e salva em Parquet

        Formato esperado da resposta da API:
            {
                "items": [{"id": 1, "name": "Ana"}, ...],
                "has_next": true,
                "total": 5000
            }

        Args:
            endpoint:    Caminho do recurso sem a URL base. Ex: "customers"
            output_path: Caminho completo onde salvar o arquivo .parquet.
            page_size:   Quantos registros buscar por página (padrão: 500).

        Returns:
            DataFrame com todos os registros coletados.
        """
        # Garante que a pasta de destino existe antes de tentar salvar.
        # parents=True: cria pastas intermediárias (como mkdir -p no Linux).
        # exist_ok=True: não lança erro se a pasta já existir.
        output_path.parent.mkdir(parents=True, exist_ok=True)

        url = f"{self.base_url}/{endpoint}"
        all_records = []
        page = 1

        # Cria o decorator de retry com os parâmetros desta instância e
        # aplica diretamente ao método _get_page.
        fetch_with_retry = self._make_retry_decorator()(self._get_page)

        logger.info("Iniciando coleta de '%s' (page_size=%d)", endpoint, page_size)

        while True:
            logger.info("  Página %d...", page)

            data = fetch_with_retry(url, {"page": page, "page_size": page_size})

            # .extend() adiciona cada item de data["items"] individualmente
            # à lista all_records. É equivalente a um loop, mas mais eficiente.
            all_records.extend(data["items"])

            # .get("has_next") é mais seguro que data["has_next"]:
            # se a chave não existir, retorna None (falsy) em vez de KeyError.
            if not data.get("has_next"):
                break

            page += 1

        # pd.DataFrame(lista_de_dicts) converte para tabela:
        # cada dict vira uma linha, as chaves viram nomes de colunas.
        df = pd.DataFrame(all_records)

        # Parquet vs CSV:
        # - Parquet preserva tipos (int permanece int, datetime permanece datetime)
        # - CSV converte tudo para string — você perde informação de tipo
        # - Parquet comprime automaticamente: ~5-10x menor que CSV equivalente
        # - Parquet lê colunas seletivamente: muito mais rápido para análise
        # index=False: não salva a coluna de índice do DataFrame (0,1,2,3...)
        df.to_parquet(output_path, index=False)

        logger.info(
            "Coleta concluída: %d registros, %d colunas → %s",
            len(df),
            len(df.columns),
            output_path,
        )
        return df


def create_collector_from_env() -> APICollector:
    """Cria um APICollector lendo as configurações das variáveis de ambiente.

    Conveniência para uso no pipeline principal.
    As variáveis são definidas no .env (nunca no código).

    Returns:
        APICollector configurado e pronto para uso.

    Raises:
        KeyError: se API_BASE_URL ou API_TOKEN não estiverem definidas.
    """
    return APICollector(
        base_url=os.environ["API_BASE_URL"],
        token=os.environ["API_TOKEN"],
        max_attempts=int(os.getenv("API_MAX_ATTEMPTS", "5")),
        min_wait_secs=float(os.getenv("API_MIN_WAIT", "1.0")),
        max_wait_secs=float(os.getenv("API_MAX_WAIT", "60.0")),
    )


---
## Parte 1.5 — Servidor Fake (demonstracao ao vivo)

Em vez de mockar a API so nos testes, vamos subir um servidor HTTP local com FastAPI  
que se comporta como uma API real — com paginacao, autenticacao e **erros propositais**.

Você vai ver no log: `WARNING — Retry acionado, tentativa 2/5`

In [6]:
# Cria o arquivo do servidor fake
# (normalmente voce ja teria este arquivo no repositorio)

fake_server_code = '''
"""Servidor fake para simular a API de clientes com erros propositais."""
import random
import uvicorn
from fastapi import FastAPI, HTTPException, Header

app = FastAPI(title="Fake Streaming API")

# Gera dados sinteticos uma vez ao subir o servidor
random.seed(42)
TOTAL_CUSTOMERS = 1500
PLANS = ["basic", "standard", "premium"]
REGIONS = ["SP", "RJ", "MG", "RS", "PR"]

FAKE_TOKEN = "token-de-teste"

ALL_CUSTOMERS = [
    {
        "customer_id": f"C{i:04d}",
        "plan": random.choice(PLANS),
        "region": random.choice(REGIONS),
        "age": random.randint(18, 70),
        "churned": random.randint(0, 1),
        "signup_date": f"202{random.randint(5,6)}-{random.randint(1,12):02d}-{random.randint(1,28):02d}",
    }
    for i in range(1, TOTAL_CUSTOMERS + 1)
]

def _maybe_fail():
    """Injeta erros aleatorios para demonstrar o retry em acao."""
    rng = random.random()
    if rng < 0.30:   # 30% de chance de HTTP 500
        raise HTTPException(status_code=500, detail="Erro simulado!")
    if rng < 0.40:   # 10% de chance de HTTP 429
        raise HTTPException(status_code=429, detail="Rate limit!")
        
def _valida_token(authorization):
    """Verifica se o token passado pelo usuário é um token válido"""
    token = authorization.replace("Bearer ", "")
    if token != FAKE_TOKEN:
        raise HTTPException(status_code=401, detail="Token invalido")

@app.get("/v1/customers")
def get_customers(
    page: int = 1,
    page_size: int = 100,
    authorization: str = Header(default=None),
):
    # Valida o token
    _valida_token(authorization)
    
    # Injeta erros aleatorios
    _maybe_fail()
    
    # Paginacao
    start = (page - 1) * page_size
    end = start + page_size
    items = ALL_CUSTOMERS[start:end]
    return {
        "items": items,
        "page": page,
        "has_next": end < len(ALL_CUSTOMERS),
    }

if __name__ == "__main__":
    uvicorn.run(app, host="127.0.0.1", port=8000)
'''

Path('scripts/fake_api_server.py').write_text(fake_server_code, encoding="utf-8")
print('Arquivo criado: scripts/fake_api_server.py')
print()
print('Para rodar:')
print('  Abra um terminal separado e execute:')
print('  python scripts/fake_api_server.py')

Arquivo criado: scripts/fake_api_server.py

Para rodar:
  Abra um terminal separado e execute:
  python scripts/fake_api_server.py


In [8]:
# Testa a coleta com o servidor fake rodando
# ATENCAO: rode este bloco SÓ depois de subir o servidor fake no terminal

import logging
from pathlib import Path

logging.basicConfig(level=logging.INFO, format='%(levelname)s - %(message)s')

collector = APICollector(
    base_url='http://127.0.0.1:8000/v1',
    token='token-de-teste',
)
df_clientes = collector.collect(
    endpoint='customers',
    output_path=Path('data/raw/customers_api.parquet'),
    page_size=100,   # a API fake pagina de 100 em 100 por padrao no exemplo antigo
)

print(f'\nClientes coletados: {len(df_clientes)}')
print(f'Colunas: {list(df_clientes.columns)}')
df_clientes.head(3)

INFO - Iniciando coleta de 'customers' (page_size=100)
INFO -   Página 1...
WARNING - Retrying __main__.APICollector._get_page in 0.83606062778918 seconds as it raised HTTPError: 500 Server Error: Internal Server Error for url: http://127.0.0.1:8000/v1/customers?page=1&page_size=100.
WARNING - Retrying __main__.APICollector._get_page in 0.712877102033227 seconds as it raised HTTPError: 500 Server Error: Internal Server Error for url: http://127.0.0.1:8000/v1/customers?page=1&page_size=100.
INFO -   Página 2...
WARNING - Retrying __main__.APICollector._get_page in 0.4634427778216127 seconds as it raised HTTPError: 500 Server Error: Internal Server Error for url: http://127.0.0.1:8000/v1/customers?page=2&page_size=100.
INFO -   Página 3...
INFO -   Página 4...
INFO -   Página 5...
WARNING - Retrying __main__.APICollector._get_page in 0.31285980998505913 seconds as it raised HTTPError: 500 Server Error: Internal Server Error for url: http://127.0.0.1:8000/v1/customers?page=5&page_size=100


Clientes coletados: 1500
Colunas: ['customer_id', 'plan', 'region', 'age', 'churned', 'signup_date']


,customer_id,plan,region,age,churned,signup_date
0,C0001,premium,SP,19,1,2025-04-05
1,C0002,premium,SP,61,0,2026-01-01
2,C0003,basic,RJ,32,0,2025-12-21
